# **FLUXO DE MODELAGEM DE PROJEÇÃO, COM VALIDAÇÃO SIMPLES + CRUZADA**
### *Passos sugeridos*
---



### **Bibliotecas**

In [ ]:
# Instalação de bibliotecas não disponíveis na instalação padrão do Python no Google Colab
# ! pip install shap

In [1]:
import pandas as pd                                                                # Manipulação de dados
import numpy as np                                                                 # Realização de cálculos específicos
import matplotlib.pyplot as plt                                                    # Visualização de dados 
import seaborn as sns                                                              # Visualização de dados
import time                                                                        # Cálculo de tempo de execução
import math                                                                        # Funções matemáticas
from scipy.stats import randint, uniform, loguniform                               # Geração de valores aleatórios
from sklearn.compose import ColumnTransformer                                      # Transformação de colunas
from sklearn.preprocessing import StandardScaler, OneHotEncoder                    # Transformação de colunas
from sklearn.base import clone                                                     # Criação de cópias de modelos
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV    # Validação simples e validação cruzada
from sklearn.metrics import r2_score, root_mean_squared_error                      # Métricas de avaliação de modelos
from sklearn.linear_model import Ridge, Lasso, ElasticNet, SGDRegressor            # Regressão linear com regularização
from sklearn.tree import DecisionTreeRegressor                                     # Árvore de regressão
from sklearn.ensemble import RandomForestRegressor                                 # Floresta aleatória
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor          # Impulsionamento tradicional
from sklearn.ensemble import HistGradientBoostingRegressor                         # Impulsionamento com histogramas
from xgboost import XGBRegressor                                                   # Impulsionamento via XGBoost
from lightgbm import LGBMRegressor                                                 # Impulsionamento via LightGBM
from sklearn.inspection import permutation_importance                              # Importância de variáveis por permutação
import shap                                                                        # Interpretabilidade de variáveis nos modelos

### **Leitura da base de dados**

In [ ]:
dados = pd.read_table("nome_da_base_de_dados.txt",  # Substitua pelo nome da base de dados
                      sep="\t",                     # Ajuste o separador de colunas, se necessário
                      decimal=".",                  # Ajuste o separador de decimal, se necessário
                      header=0)

### **Visualização da base de dados**

In [ ]:
dados.head()

### **Dimensões da base de dados**

In [ ]:
dados.shape

### **Tipos das colunas da base de dados**

In [ ]:
dados.dtypes

### **Etapa 1: Tratamento de valores ausentes, especificação de variáveis e análise bivariada**

*Tratamento de valores ausentes nas variáveis explicativas quantitativas, se houver*

In [ ]:
# Verificando a quantidade de valores ausentes
dados.isna().sum()

In [ ]:
# Sugestão de tratamento de valores ausentes em uma variável explicativa quantitativa, por meio da criação de faixas
# dados['NOME_VARIAVEL'] = pd.to_numeric(dados['NOME_VARIAVEL'])
# dados['NOME_VARIAVEL_CAT'] = pd.qcut(dados['NOME_VARIAVEL'], q=10, duplicates='drop')
# dados['NOME_VARIAVEL_CAT'] = dados['NOME_VARIAVEL_CAT'].cat.add_categories('NA').fillna('NA')
# dados['NOME_VARIAVEL_CAT'].value_counts(dropna=False)

*Lista de nomes das variáveis explicativas, separando em quantitativas e qualitativas*

In [ ]:
# Variáveis explicativas quantitativas (deixar vazio [] caso não haja nenhuma)
lista_X_quanti = ['NOME_VARIAVEL_1',
                  'NOME_VARIAVEL_2',
                  'NOME_VARIAVEL_3',
                  '...']

# Variáveis explicativas qualitativas (deixar vazio [] caso não haja nenhuma)
lista_X_quali = ['NOME_VARIAVEL_1',
                 'NOME_VARIAVEL_2',
                 'NOME_VARIAVEL_3',
                 '...']

*Objetos para variável resposta (y) e explicativas (X)*

In [ ]:
y = dados['NOME_VARIAVEL_RESPOSTA']
X = dados[lista_X_quanti + lista_X_quali]

*Análise bivariada: gráficos de boxplot para variáveis explicativas qualitativas versus variável resposta*

In [ ]:
if lista_X_quali:
    n = len(lista_X_quali)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows=nrows,
                             ncols=ncols,
                             figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for i, var in enumerate(lista_X_quali):
        sns.boxplot(y=dados[var],
                    x=y,
                    ax=axes[i],
                    orient='h',
                    color='darkturquoise')
        axes[i].set_title(f'{y.name} vs. {var}')
        axes[i].tick_params(axis='x', rotation=0)
        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout(h_pad=2, w_pad=2)
    plt.show()

*Análise bivariada: gráficos de dispersão para variáveis explicativas quantitativas versus variável resposta*

In [ ]:
if lista_X_quanti:
    n = len(lista_X_quanti)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows=nrows,
                             ncols=ncols,
                             figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for i, var in enumerate(lista_X_quanti):
        sns.scatterplot(x=dados[var],
                        y=y,
                        ax=axes[i],
                        color='darkturquoise',
                        alpha=0.5)
        axes[i].set_title(f'{y.name} vs. {var}')
        axes[i].set_ylabel("")

    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout()
    plt.show()

### **Etapa 2: Divisão de treino e teste externo**

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y,
                                                        test_size=0.2,
                                                        random_state=123)

### **Etapa 3: Pré-processamento de variáveis explicativas**

*Definição das funções de pré-processamento: Padronização das variáveis quantitativas + Codificação one-hot para variáveis qualitativas*

In [ ]:
preprocessador = ColumnTransformer(transformers=[
    ("quanti", StandardScaler(), lista_X_quanti),
    ("quali", OneHotEncoder(sparse_output=False, drop="first", handle_unknown='ignore'), lista_X_quali)
])

*Criação do pré-processamento no conjunto de treino* <br>

In [ ]:
X_treino_tratada = preprocessador.fit_transform(X_treino)  # O pré-processamento não deve envolver o conjunto de teste externo, que deve ficar reservado apenas para aplicações de resultados já obtidos, e nunca construções
if lista_X_quali:
    nomes_quali = list(preprocessador.named_transformers_['quali'].get_feature_names_out(lista_X_quali))
else:
    nomes_quali = []
nomes_variaveis = list(lista_X_quanti) + nomes_quali

In [ ]:
X_treino_tratada = pd.DataFrame(X_treino_tratada, columns=nomes_variaveis)
X_treino_tratada.head()

*Aplicação do pré-processamento no conjunto de teste externo*

In [ ]:
X_teste_tratada = preprocessador.transform(X_teste)  # Note que o mesmo objeto de pré-processamento deve ser utilizado, agora com 'transform' em vez de 'fit_transform'
X_teste_tratada = pd.DataFrame(X_teste_tratada, columns=nomes_variaveis)
X_teste_tratada.head()

### **Etapa 4: Definição dos algoritmos e intervalos de busca de hiperparâmetros**

In [ ]:
algoritmos = {

    'REGRESSÃO LINEAR + RIDGE': (Ridge(), {
        'alpha': loguniform(1e-5, 1e1)                # Peso da penalização
    }),

    'REGRESSÃO LINEAR + LASSO': (Lasso(random_state=123, max_iter=1000), {
        'alpha': loguniform(1e-5, 1e1)                # Peso da penalização
    }),

    'REGRESSÃO LINEAR + ELASTICNET': (ElasticNet(random_state=123, max_iter=1000), {
        'alpha': loguniform(1e-5, 1e1),               # Peso da penalização
        'l1_ratio': uniform(0, 1),                    # Contribuição da penalização LASSO (L1) em relação a Ridge (L2)
    }),

    'REGRESSÃO LINEAR VIA GRADIENTE DESCENDENTE': (SGDRegressor(random_state=123, max_iter=100000, tol=1e-3), {
        'penalty': ['l2', 'l1', 'elasticnet'],
        'alpha': loguniform(1e-5, 1e1),               # Peso da penalização
        'l1_ratio': uniform(0, 1),                    # Contribuição da penalização LASSO (L1) em relação a Ridge (L2)
        'eta0': loguniform(1e-4, 1e-1),               # Taxa de aprendizado inicial
        'learning_rate': ['constant',                 # Atualização de eta0: manter sempre constante
                          'invscaling',               # Atualização de eta0: reduzir gradativamente ao longo das iterações
                          'adaptive']                 # Atualização de eta0: reduzir gradativamente ao longo das iterações, se não houver redução do erro
    }),

    'ÁRVORE DE REGRESSÃO': (DecisionTreeRegressor(random_state=123), {
        'min_impurity_decrease': uniform(0, 0.05),    # Percentual mínimo de redução de impureza
        'min_samples_split': randint(200, 500),       # Qtde. mínima de observações no nó "pai" (ATENÇÃO: depende do tamanho da base)
        'min_samples_leaf': randint(50, 100),         # Qtde. mínima de observações no nó "filho" (ATENÇÃO: depende do tamanho da base)
        'max_depth': randint(2, 10),                  # Profundidade máxima da árvore
    }),

    'FLORESTA ALEATÓRIA': (RandomForestRegressor(random_state=123, n_jobs = -1), {
        'n_estimators': randint(10, 500),             # Qtde. de árvores na floresta
        'min_impurity_decrease': uniform(0, 0.05),    # Percentual mínimo de redução de impureza
        'min_samples_split': randint(200, 500),       # Qtde. mínima de observações no nó "pai" (ATENÇÃO: depende do tamanho da base)
        'min_samples_leaf': randint(50, 100),         # Qtde. mínima de observações no nó "filho" (ATENÇÃO: depende do tamanho da base)
        'max_samples': uniform(0.6, 0.4),             # Percentual de observações consideradas em cada árvore
        'max_features': ['sqrt', 'log2', None],       # Qtde. de variáveis consideradas em cada árvore
        'max_depth': randint(2, 10)                   # Profundidade máxima das árvores
    }),

    'ADABOOST': (AdaBoostRegressor(random_state=123), {
        'n_estimators': randint(10, 300),             # Qtde. de árvores no boosting
        'learning_rate': loguniform(1e-4, 1e-1)       # Taxa de aprendizado
    }),

    'GRADIENT BOOSTING': (GradientBoostingRegressor(random_state=123), {
        'n_estimators': randint(10, 300),             # Qtde. de árvores no boosting
        'min_impurity_decrease': uniform(0, 0.05),    # Percentual mínimo de redução de impureza
        'learning_rate': loguniform(1e-4, 1e-1),      # Taxa de aprendizado
        'min_samples_split': randint(200, 500),       # Qtde. mínima de observações no nó "pai" (ATENÇÃO: depende do tamanho da base)
        'min_samples_leaf': randint(50, 100),         # Qtde. mínima de observações no nó "filho" (ATENÇÃO: depende do tamanho da base)
        'subsample': uniform(0.6, 0.4),               # Percentual de observações consideradas em cada árvore
        'max_features': ['sqrt', 'log2', None],       # Qtde. de variáveis consideradas em cada árvore
        'max_depth': randint(2, 10)                   # Profundidade máxima das árvores
    }),

    'GRADIENT BOOSTING COM HISTOGRAMAS': (HistGradientBoostingRegressor(random_state=123), {
        'max_iter': randint(10, 300),                 # Qtde. de árvores no boosting
        'learning_rate': loguniform(1e-4, 1e-1),      # Taxa de aprendizado
        'min_samples_leaf': randint(50, 100),         # Qtde. mínima de observações no nó "filho" (ATENÇÃO: depende do tamanho da base)
        'max_leaf_nodes': randint(10, 30),            # Qtde. máxima de nós finais (ATENÇÃO: depende do tamanho da base)
        'max_bins': randint(10, 255),                 # Qtde. máxima de bins para discretização
        'l2_regularization': loguniform(1e-5, 1e2),   # Contribuição da penalização Ridge (L2)
        'max_depth': randint(2, 10)                   # Profundidade máxima das árvores
    }),

    'XGBOOST': (XGBRegressor(random_state=123, n_jobs=-1), {
        'n_estimators': randint(10, 300),             # Qtde. de árvores no boosting
        'learning_rate': loguniform(1e-4, 1e-1),      # Taxa de aprendizado
        'colsample_bytree': uniform(0.5, 0.5),        # Percentual de variáveis consideradas em cada árvore
        'subsample': uniform(0.6, 0.4),               # Percentual de observações consideradas em cada árvore
        'gamma': uniform(0, 5),                       # Redução mínima na função de perda para realizar uma quebra
        'reg_alpha': loguniform(1e-5, 1e2),           # Contribuição da penalização LASSO (L1)
        'reg_lambda': loguniform(1e-5, 1e2),          # Contribuição da penalização Ridge (L2)
        'max_depth': randint(2, 10)                   # Profundidade máxima das árvores
    }),

    'LIGHTGBM': (LGBMRegressor(random_state=123, n_jobs=-1, verbosity=-1), {
        'n_estimators': randint(10, 300),             # Qtde. de árvores no boosting
        'learning_rate': loguniform(1e-4, 1e-1),      # Taxa de aprendizado
        'min_child_samples': randint(50, 100),        # Qtde. mínima de observações no nó "filho" (ATENÇÃO: depende do tamanho da base)
        'num_leaves': randint(10, 30),                # Qtde. máxima de nós finais (ATENÇÃO: depende do tamanho da base)
        'subsample': uniform(0.6, 0.4),               # Percentual de observações consideradas em cada árvore
        'reg_alpha': loguniform(1e-5, 1e2),           # Contribuição da penalização LASSO (L1)
        'reg_lambda': loguniform(1e-5, 1e2),          # Contribuição da penalização Ridge (L2)
        'max_depth': randint(2, 10)                   # Profundidade máxima das árvores
    })
    
}

### **Etapa 5: Construção de modelos usando validação cruzada, busca aleatória e teste externo**

*Estratégia de validação cruzada (k-fold) e quantidade máxima de iterações de busca*

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
qtd_iteracoes = 100

*Construção dos modelos*

In [ ]:
resultados = []         # Lista para armazenar resultados
contador_modelos = 0    # Contador de modelos testados

for nome_algoritmo, (classe_algoritmo, hiperparametros) in algoritmos.items():

    # Início da contagem de tempo e de modelos
    inicio = time.time()

    # Busca aleatória de hiperparâmetros com validação cruzada (randomized search)
    busca = RandomizedSearchCV(
        estimator=classe_algoritmo,
        param_distributions=hiperparametros,
        n_iter=1 if not hiperparametros else qtd_iteracoes,
        scoring='neg_root_mean_squared_error',
        cv=kf,
        return_train_score=True,
        random_state=123,
        n_jobs=-1
    )

    # Ajuste dos modelos
    busca.fit(X_treino_tratada, y_treino)

    # Armazenar os resultados da busca
    resultados_aux = pd.DataFrame({
        "num_modelo": [f"Modelo {contador_modelos + i}" for i in range(len(busca.cv_results_["params"]))],
        "nome_algoritmo": nome_algoritmo,
        "classe_algoritmo": type(classe_algoritmo).__name__,
        "hiperparametros": busca.cv_results_["params"],
        "media_erro_treino": -busca.cv_results_["mean_train_score"],
        "media_erro_teste_interno": -busca.cv_results_["mean_test_score"],
        "dp_erro_treino": busca.cv_results_["std_train_score"],
        "dp_erro_teste_interno": busca.cv_results_["std_test_score"]
    })

    # Calcular erro no teste externo, para cada modelo
    erro_teste = []
    for params in resultados_aux["hiperparametros"]:
        estimador = clone(classe_algoritmo).set_params(**params)
        estimador.fit(X_treino_tratada, y_treino)
        y_hat_teste = estimador.predict(X_teste_tratada)
        erro = root_mean_squared_error(y_teste, y_hat_teste)
        erro_teste.append(erro)

    # Adicionar coluna de erro em teste externo e colunas de variações absolutas de erro em relação ao treino
    resultados_aux["erro_teste_externo"] = erro_teste
    resultados_aux["var_abs_media_erro_teste_interno"] = abs(resultados_aux["media_erro_teste_interno"] - resultados_aux["media_erro_treino"])
    resultados_aux["var_abs_perc_media_erro_teste_interno"] = resultados_aux["var_abs_media_erro_teste_interno"] / resultados_aux["media_erro_treino"]
    resultados_aux["var_abs_erro_teste_externo"] = abs(resultados_aux["erro_teste_externo"] - resultados_aux["media_erro_treino"])
    resultados_aux["var_abs_perc_erro_teste_externo"] = resultados_aux["var_abs_erro_teste_externo"] / resultados_aux["media_erro_treino"]

    resultados.append(resultados_aux)
    contador_modelos += len(resultados_aux)

    # Fim da contagem de tempo e mensagem de finalização
    fim = time.time()
    print(f"({math.ceil(contador_modelos/qtd_iteracoes)} de {len(algoritmos)}) Processo concluído para o algoritmo {nome_algoritmo} em {fim - inicio:.1f} segundos!")

# Criação de data frame com os resultados
resultados = pd.concat(resultados, ignore_index=True)

### **Etapa 6: Comparação de modelos**

*Gráficos de resumo dos algoritmos*

In [ ]:
# Gráfico do erro médio nos conjuntos de teste interno, em média ao longo das iterações do randomized search
resumo_algoritmos = (
    resultados
    .groupby("nome_algoritmo")["media_erro_teste_interno"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
plt.barh(resumo_algoritmos.index, resumo_algoritmos.values, color="darkturquoise")
plt.title("Erro Médio de Teste Interno, em Média ao Longo das Iterações")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico do erro no conjunto de teste externo, em média ao longo das iterações do randomized search
resumo_algoritmos = (
    resultados
    .groupby("nome_algoritmo")["erro_teste_externo"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
plt.barh(resumo_algoritmos.index, resumo_algoritmos.values, color="darkturquoise")
plt.title("Erro de Teste Externo, em Média ao Longo das Iterações")
plt.tight_layout()
plt.show()

*Ordenação e filtragem dos melhores modelos*

In [ ]:
# Escolha por qual coluna ordenar
ordenar_por = "media_erro_teste_interno"
# ordenar_por = "var_abs_media_erro_teste_interno"
# ordenar_por = "var_abs_erro_teste_externo"

# Escolha quais o percentuais máximos de variação de erro (da média do treino para a média do teste interno; e da média do treino para o teste externo)
max_var_perc_erro_teste_interno = 0.1  # ex.: 0.1 representa 10%
max_var_perc_erro_teste_externo = 0.1

# Ordenação e filtragem
resultados_filt_ord = resultados[(resultados['var_abs_perc_media_erro_teste_interno'] < max_var_perc_erro_teste_interno) & (resultados['var_abs_perc_erro_teste_externo'] < max_var_perc_erro_teste_externo)]
resultados_filt_ord = resultados_filt_ord.sort_values(ordenar_por, ascending=True)

*Exibição dos melhores modelos, em formato de tabela e gráfico*

In [ ]:
pd.set_option('display.float_format', '{:.3f}'.format)   # Quantidade de casas decimais
resultados_filt_ord[['num_modelo',
                     'nome_algoritmo',
                     'hiperparametros',                     
                     'media_erro_treino',
                     'dp_erro_treino',
                     'media_erro_teste_interno',
                     'dp_erro_teste_interno',
                     'var_abs_media_erro_teste_interno',
                     'var_abs_perc_media_erro_teste_interno',
                     'erro_teste_externo',
                     'var_abs_erro_teste_externo',
                     'var_abs_perc_erro_teste_externo']].head(10)   # Escolha quantos modelos quer exibir

In [ ]:
dados_grafico = resultados_filt_ord.head(15)    # Escolha quantos modelos quer exibir
x_pos = np.arange(len(dados_grafico))           # Posições dos rótulos do eixo X
largura_barra = 0.25                            # Largura das barras

plt.figure(figsize=(18, 6))
plt.bar(x_pos - largura_barra,                  # Série com erro médio de treino
        dados_grafico["media_erro_treino"],
        width=largura_barra,
        label="Erro Médio de Treino",
        color="lightgray")
plt.bar(x_pos,                                  # Série com erro médio de teste interno
        dados_grafico["media_erro_teste_interno"],
        width=largura_barra,
        label="Erro Médio de Teste Interno",
        color="darkcyan")
plt.bar(x_pos + largura_barra,                  # Série com erro médio de teste externo
        dados_grafico["erro_teste_externo"],
        width=largura_barra,
        label="Erro de Teste Externo",
        color="turquoise")

plt.xticks(x_pos,
           dados_grafico["num_modelo"],
           rotation=0,
           ha='center')
plt.ylabel("Erro")
plt.title("Comparativo dos Melhores Modelos: Treino, Teste Interno e Teste Externo")
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

### **Etapa 7: Escolha do modelo**

*Escolha do modelo*

In [ ]:
num_modelo_escolhido = "Modelo XXX"   # Substitua aqui o número do modelo escolhido

*Características do modelo escolhido*

In [ ]:
linha = resultados.loc[resultados["num_modelo"] == num_modelo_escolhido].squeeze()
nome_algoritmo = linha["nome_algoritmo"]
hiperparametros = linha["hiperparametros"]
classe = algoritmos[nome_algoritmo][0]
print(f"O modelo final adotado foi construído com o algoritmo {nome_algoritmo}, com hiperparâmetros: { {k: round(v, 4) if isinstance(v, float) else v for k, v in hiperparametros.items()} }")

*Retreino do modelo no conjunto de treino completo*

In [ ]:
modelo_final = classe.set_params(**hiperparametros)
modelo_final.fit(X_treino_tratada, y_treino)

### **Etapa 8: Análises adicionais no conjunto de teste externo**

*Aplicação do modelo final no conjunto de teste externo*

In [ ]:
y_hat_teste = modelo_final.predict(X_teste_tratada)

*Erro quadrático médio em raiz (RMSE)*

In [ ]:
np.sqrt(np.mean((y_teste - y_hat_teste)**2))

*Erro absoluto médio (MAE)*

In [ ]:
np.mean(np.abs(y_teste - y_hat_teste))

*Erro absoluto percentual médio (MAPE)*

In [ ]:
np.mean(np.abs((y_teste - y_hat_teste) / y))

*Coeficiente de determinação (R²)*

In [ ]:
r2_score(y_teste, y_hat_teste)

*Resíduos do modelo (e)*

In [ ]:
resid = y_teste - y_hat_teste

*Histograma dos resíduos*

In [ ]:
sns.histplot(resid,
             color="darkturquoise",
             edgecolor="white",
             kde=True)
plt.title("Histograma dos Resíduos")
plt.xlabel("Resíduos")
plt.ylabel("Frequência")
plt.show()

*Gráfico de resíduos vs. valores preditos*

In [ ]:
plt.scatter(y_hat_teste,
            resid,
            color="darkturquoise",
            alpha=0.4)
plt.axhline(0, color='red', linestyle='--')
plt.title("Resíduos vs. Valores Preditos")
plt.xlabel("Valores Preditos")
plt.ylabel("Resíduos")
plt.show()

*Gráfico de valores observados vs. valores preditos da resposta*

In [ ]:
min_eixo = min(y_teste.min(), y_hat_teste.min())
max_eixo = max(y_teste.max(), y_hat_teste.max())
plt.scatter(y_teste,
            y_hat_teste,
            color="darkturquoise",
            alpha=0.4)
plt.plot([min_eixo, max_eixo], [min_eixo, max_eixo], color='red', linestyle='--')
plt.title("Valores Observados vs. Valores Preditos")
plt.xlabel("Valores Observados")
plt.ylabel("Valores Preditos")
plt.show()

### **Etapa 9: Interpretabilidade do modelo final**

*Cálculo da importância em redução de impureza para cada variável no modelo final (aplicável apenas para modelos baseados em árvores)*

In [ ]:
# ATENÇÃO! Aplicável apenas caso o modelo final seja baseado em árvores; caso contrário, resultará em erro
importancias_impureza = pd.Series(modelo_final.feature_importances_,
                                  index=X_treino_tratada.columns).sort_values()

*Gráfico de importâncias em redução de impureza (aplicável apenas para modelos baseados em árvores)*

In [ ]:
# ATENÇÃO! Aplicável apenas caso o modelo final seja baseado em árvores; caso contrário, resultará em erro
importancias_impureza.plot.barh(title="Importância das Variáveis por Redução de Impureza",
                                  color="darkturquoise")
plt.xlabel("Redução Média de Impureza nas Quebras")
plt.tight_layout()
plt.show()

*Cálculo da importância via permutação para cada variável no modelo final*

In [ ]:
importancias_permutacao = permutation_importance(modelo_final,
                                                 X_treino_tratada,
                                                 y_treino,
                                                 n_repeats=30,
                                                 random_state=0, 
                                                 scoring='neg_root_mean_squared_error')
importancias_permutacao = pd.Series(importancias_permutacao.importances_mean,
                                    index=X_treino_tratada.columns).sort_values()

*Gráfico de importâncias por permutação*

In [ ]:
importancias_permutacao.plot.barh(title="Importância das Variáveis por Permutação",
                                  color="darkturquoise")
plt.xlabel("Impacto Médio na Medida de Erro")
plt.tight_layout()
plt.show()

*Cálculo dos valores SHAP das variáveis no modelo final*

In [ ]:
if nome_algoritmo not in ['ADABOOST']:
    # Caso demore para rodar, especialmente em bases grandes, substitua X_treino_tratada.shape[0] por um número de tamanho de subamostra (ex.: 100)
    X_amostra = shap.sample(X_treino_tratada, X_treino_tratada.shape[0], random_state=123)
    explainer = shap.Explainer(modelo_final, X_amostra)
    valores_shap = explainer(X_amostra, check_additivity=False)
else:
    # Para AdaBoost, a implementação do cálculo de valores SHAP não é direta; é preciso usar um método mais oneroso computacionalmente, em uma subamostra pequena de observações
    X_amostra = shap.sample(X_treino_tratada, 100, random_state=123)
    explainer = shap.Explainer(modelo_final.predict, X_amostra)
    valores_shap = explainer(X_amostra, check_additivity=False)

*Gráfico de valores absolutos médios de SHAP*

In [ ]:
shap.plots.bar(valores_shap)

*Gráfico de direção do impacto baseado em SHAP (beeswarm)*

In [ ]:
shap.summary_plot(valores_shap, X_amostra, max_display=10)

### **Etapa 10: Utilização do modelo em nova base de dados**

*Leitura da nova base de dados*

In [ ]:
# Considerando a própria base de construção, mas pode ser substituída por uma nova, se houver
dados_novos = pd.read_table("nome_da_nova_base_de_dados.txt",  # Substitua pelo nome da base de dados
                            sep="\t",                          # Ajuste o separador de colunas, se necessário
                            dec=".",                           # Ajuste o separador de decimal, se necessário
                            header=0)

*Verificação de compatibilidade de nomes e tipos das variáveis, em relação à base de construção*

In [ ]:
# Comparação de nomes
dados.columns == dados_novos.columns

In [ ]:
# Comparação de tipos
dados.dtypes == dados_novos.dtypes

*Criação de objetos y e X*

In [ ]:
y_nova = dados['NOME_VARIAVEL_RESPOSTA']
X_nova = dados[lista_X_quanti + lista_X_quali]

*Aplicação do pré-processamento e criação de matriz X tratada*

In [ ]:
X_tratada_nova = preprocessador.transform(X)  # Note que o mesmo objeto de pré-processamento deve ser utilizado, agora com 'transform' em vez de 'fit_transform'
X_tratada_nova = pd.DataFrame(X_tratada_nova, columns=nomes_variaveis)

*Aplicação do modelo final na nova base de dados*

In [ ]:
y_hat_nova = modelo_final.predict(X_tratada_nova)